# Product Sales Analysis - July 2026

## 1. Setup
Import libraries and select the reporting month.

In [372]:
import pandas as pd
from pathlib import Path

In [373]:
SELECTED_FOLDER = '07.2026'

## 2. Load Sales
Combine daily CSV exports and extract dates from filenames.

In [374]:
def load_data(file_path: str) -> pd.DataFrame:
    frames = []
    try:
        for files in Path(file_path).glob('*.csv'):
            df = pd.read_csv(files)
            report_date = pd.to_datetime(files.stem, format='%Y.%m.%d')
            df['report_date'] = report_date
            frames.append(df)
    except FileNotFoundError:
        print(f"File not found: {file_path}")
        return pd.DataFrame()

    return pd.concat(frames, ignore_index=False).sort_values(by='report_date', ignore_index=True)

In [375]:
source_dir = Path.cwd().parent / 'data' / 'raw' / SELECTED_FOLDER

In [376]:
sales_raw = load_data(source_dir)

In [378]:
sales_raw

,Typ,Nazwa przedmiotu,Sprzedany,Cena,Okres,Location,report_date
0,Pozycja,Futomaki ŁOSOŚ GRILLOWANY | 6 szt,12,456.0,01.07.2026-01.07.2026,VIBE SUSHI,2026-07-01
1,Dodatek,Sos spicy majo 30ml,1,NaN,01.07.2026-01.07.2026,VIBE SUSHI,2026-07-01
2,Dodatek,Wasabi,1,2.0,01.07.2026-01.07.2026,VIBE SUSHI,2026-07-01
3,Dodatek,Krabs - Burger,1,25.0,01.07.2026-01.07.2026,VIBE SUSHI,2026-07-01
4,Dodatek,Sos teriyaki 30ml,1,3.0,01.07.2026-01.07.2026,VIBE SUSHI,2026-07-01
...,...,...,...,...,...,...,...
825,Pozycja,Zapieczona rolka ŁOSOŚ SUROWY | 8 szt,1,60.0,31.07.2026-31.07.2026,VIBE SUSHI,2026-07-31
826,Pozycja,Fuji roll | 8 szt,1,65.0,31.07.2026-31.07.2026,VIBE SUSHI,2026-07-31
827,Pozycja,Zapieczona rolka FELIX | 8 szt (NOWOŚĆ),1,60.0,31.07.2026-31.07.2026,VIBE SUSHI,2026-07-31
828,Pozycja,California KRAB-MIX | 8 szt,2,90.0,31.07.2026-31.07.2026,VIBE SUSHI,2026-07-31


## 3. Clean Sales
Rename columns, remove incomplete rows, and exclude ambiguous size modifiers.

In [379]:
sales_raw = sales_raw.drop(['Okres', 'Location'], axis=1)

In [380]:
sales_raw

,Typ,Nazwa przedmiotu,Sprzedany,Cena,report_date
0,Pozycja,Futomaki ŁOSOŚ GRILLOWANY | 6 szt,12,456.0,2026-07-01
1,Dodatek,Sos spicy majo 30ml,1,NaN,2026-07-01
2,Dodatek,Wasabi,1,2.0,2026-07-01
3,Dodatek,Krabs - Burger,1,25.0,2026-07-01
4,Dodatek,Sos teriyaki 30ml,1,3.0,2026-07-01
...,...,...,...,...,...
825,Pozycja,Zapieczona rolka ŁOSOŚ SUROWY | 8 szt,1,60.0,2026-07-31
826,Pozycja,Fuji roll | 8 szt,1,65.0,2026-07-31
827,Pozycja,Zapieczona rolka FELIX | 8 szt (NOWOŚĆ),1,60.0,2026-07-31
828,Pozycja,California KRAB-MIX | 8 szt,2,90.0,2026-07-31


In [381]:
sales_raw = sales_raw.rename({
    'Typ': 'product_type',
    'Nazwa przedmiotu': 'product_name',
    'Sprzedany': 'total_sales',
    'Cena': 'total_revenue',
    'report_date': 'date',
}, axis=1)

In [382]:
sales_raw

,product_type,product_name,total_sales,total_revenue,date
0,Pozycja,Futomaki ŁOSOŚ GRILLOWANY | 6 szt,12,456.0,2026-07-01
1,Dodatek,Sos spicy majo 30ml,1,NaN,2026-07-01
2,Dodatek,Wasabi,1,2.0,2026-07-01
3,Dodatek,Krabs - Burger,1,25.0,2026-07-01
4,Dodatek,Sos teriyaki 30ml,1,3.0,2026-07-01
...,...,...,...,...,...
825,Pozycja,Zapieczona rolka ŁOSOŚ SUROWY | 8 szt,1,60.0,2026-07-31
826,Pozycja,Fuji roll | 8 szt,1,65.0,2026-07-31
827,Pozycja,Zapieczona rolka FELIX | 8 szt (NOWOŚĆ),1,60.0,2026-07-31
828,Pozycja,California KRAB-MIX | 8 szt,2,90.0,2026-07-31


In [383]:
sales = sales_raw.dropna()

In [384]:
sales

,product_type,product_name,total_sales,total_revenue,date
0,Pozycja,Futomaki ŁOSOŚ GRILLOWANY | 6 szt,12,456.0,2026-07-01
2,Dodatek,Wasabi,1,2.0,2026-07-01
3,Dodatek,Krabs - Burger,1,25.0,2026-07-01
4,Dodatek,Sos teriyaki 30ml,1,3.0,2026-07-01
5,Dodatek,Sos sojowy 30ml,2,6.0,2026-07-01
...,...,...,...,...,...
824,Pozycja,Futomaki KREWETKA PANKO | 6 szt,1,40.0,2026-07-31
825,Pozycja,Zapieczona rolka ŁOSOŚ SUROWY | 8 szt,1,60.0,2026-07-31
826,Pozycja,Fuji roll | 8 szt,1,65.0,2026-07-31
827,Pozycja,Zapieczona rolka FELIX | 8 szt (NOWOŚĆ),1,60.0,2026-07-31


In [385]:
sales.columns[sales.isna().any()].to_list() # There are no NaN columns.

[]

In [386]:
sales = sales.drop('product_type', axis=1)

In [387]:
sales = sales[sales['product_name'] != '12 szt'] # Exclude size modifiers that cannot be linked to a specific product.

## 4. Prepare Product Mapping
List sales names and prepare the ingredient cost and VAT reference data.

In [389]:
sorted(sales['product_name'].unique().tolist())

['Atlantic Wave roll | 8 szt',
 'Awokado fresh roll | 8 szt',
 'Black dragon roll | 8 szt',
 'California KRAB-MIX | 8 szt',
 'California KREWETKA GOTOWANA | 8 szt',
 'California KREWETKA PANKO | 8 szt',
 'California RAINBOW | 8 szt',
 'California SHIITAKE | 8 szt',
 'California TUŃCZYK SUROWY | 8 szt',
 'California Warzywa w tempurze | 8 szt',
 'California z czarnym ryżem ŁOSOŚ SUROWY | 8 szt',
 'California ŁOSOŚ GRILLOWANY | 8 szt',
 'California ŁOSOŚ SUROWY | 8 szt',
 'Canada roll | 8 szt',
 'Coca cola 0.5l',
 'Coca cola zero 0.5l',
 'Duo set | 28 szt',
 'Fanta jagoda 0.33l',
 'Fanta truskawka 0.33l',
 'Feliks roll | 8 szt',
 'Feliks z łososiem | 8 szt',
 'Frytki z batatów (bez sosu)',
 'Frytki z batatów (z sosem spicy - majo)',
 'Frytki z batatów (z sosem śliwkowym)',
 'Fuji roll | 8 szt',
 'Futomaki AWOKADO | 6 szt',
 'Futomaki FIRE HARMONY | 6 szt',
 'Futomaki FIRE HARMONY | 6 szt / 12 szt',
 'Futomaki FRESH ROLL | 6 szt',
 'Futomaki FRESH ROLL | 6 szt / 12 szt',
 'Futomaki GRZYBY

In [390]:
product_names = pd.DataFrame({'product_name_from_report': sorted(sales['product_name'].unique().tolist())})

In [391]:
product_names

,product_name_from_report
0,Atlantic Wave roll | 8 szt
1,Awokado fresh roll | 8 szt
2,Black dragon roll | 8 szt
3,California KRAB-MIX | 8 szt
4,California KREWETKA GOTOWANA | 8 szt
...,...
137,Zestaw Sakura | 62 szt
138,Zestaw Startowy | 16 szt
139,Zestaw Vibe | 32 szt
140,Zestaw Yami | 38 szt


In [392]:
my_calculations = pd.read_csv(Path.cwd().parent / 'data' / 'mappings' / '07.2026' / 'my_calculations.csv')

In [393]:
my_calculations = my_calculations[['ПОЗИЦИЯ', 'РЕСТ', 'КАТЕГОРИЯ', 'РАСХОДЫ', 'VAT ПРОЦ']]

In [394]:
my_calculations = my_calculations[my_calculations['РЕСТ'] == 'Vibe'].sort_values(by='ПОЗИЦИЯ')

In [395]:
my_calculations

,ПОЗИЦИЯ,РЕСТ,КАТЕГОРИЯ,РАСХОДЫ,VAT ПРОЦ
111,Akami Tamago roll,Vibe,ROLKI AUTORSKIE,13.43,8.0
104,Atlantic Wave,Vibe,ROLKI AUTORSKIE,13.87,23.0
116,Awokado delux,Vibe,ROLKI AUTORSKIE,12.37,8.0
38,Awokado fresh roll,Vibe,WEGE MENU,9.53,8.0
60,B. Wege,Vibe,SUSHI BURGER,8.54,8.0
...,...,...,...,...,...
77,ZR Surowy,Vibe,ROLKI ZAPIECZONE,12.11,8.0
79,ZR Łosoś pieczony,Vibe,ROLKI ZAPIECZONE,10.58,8.0
27,Zestaw Amour | 36 szt,Vibe,ZESTAWY,42.75,8.0
218,Zestaw Philadelphia + Krabs Burger -50%,Vibe,SUSHI BURGER,50.30,8.0


In [396]:
my_calculations = my_calculations.drop('РЕСТ', axis=1)

In [397]:
my_calculations = my_calculations.rename({
    'ПОЗИЦИЯ': 'name_from_my_calculations',
    'КАТЕГОРИЯ': 'product_category_from_my_calculations',
    'РАСХОДЫ': 'product_cost_my_calculations',
    'VAT ПРОЦ': 'vat_rate_my_calculations',
}, axis=1)

In [398]:
my_calculations = my_calculations.sort_values(by='name_from_my_calculations', ignore_index=True)

In [399]:
my_calculations

,name_from_my_calculations,product_category_from_my_calculations,product_cost_my_calculations,vat_rate_my_calculations
0,Akami Tamago roll,ROLKI AUTORSKIE,13.43,8.0
1,Atlantic Wave,ROLKI AUTORSKIE,13.87,23.0
2,Awokado delux,ROLKI AUTORSKIE,12.37,8.0
3,Awokado fresh roll,WEGE MENU,9.53,8.0
4,B. Wege,SUSHI BURGER,8.54,8.0
...,...,...,...,...
126,ZR Surowy,ROLKI ZAPIECZONE,12.11,8.0
127,ZR Łosoś pieczony,ROLKI ZAPIECZONE,10.58,8.0
128,Zestaw Amour | 36 szt,ZESTAWY,42.75,8.0
129,Zestaw Philadelphia + Krabs Burger -50%,SUSHI BURGER,50.30,8.0


### Mapping Template
Create a draft for manual matching; the reviewed mapping is loaded separately below.

In [400]:
product_costs_mapping = pd.DataFrame(
    data={
        'product_name_from_report': product_names['product_name_from_report'],
        'product_name_from_calculations_accepted': '',
        'product_category_accepted': '',
        'vat_rate_accepted': '',
        'product_cost_accepted': '',
        'name_from_my_calculations': my_calculations['name_from_my_calculations'],
        'product_category_from_my_calculations': my_calculations['product_category_from_my_calculations'],
        'product_cost_my_calculations': my_calculations['product_cost_my_calculations'],
        'vat_rate_my_calculations': my_calculations['vat_rate_my_calculations'],
    }
)

In [401]:
product_costs_mapping

,product_name_from_report,product_name_from_calculations_accepted,product_category_accepted,vat_rate_accepted,product_cost_accepted,name_from_my_calculations,product_category_from_my_calculations,product_cost_my_calculations,vat_rate_my_calculations
0,Atlantic Wave roll | 8 szt,,,,,Akami Tamago roll,ROLKI AUTORSKIE,13.43,8.0
1,Awokado fresh roll | 8 szt,,,,,Atlantic Wave,ROLKI AUTORSKIE,13.87,23.0
2,Black dragon roll | 8 szt,,,,,Awokado delux,ROLKI AUTORSKIE,12.37,8.0
3,California KRAB-MIX | 8 szt,,,,,Awokado fresh roll,WEGE MENU,9.53,8.0
4,California KREWETKA GOTOWANA | 8 szt,,,,,B. Wege,SUSHI BURGER,8.54,8.0
...,...,...,...,...,...,...,...,...,...
137,Zestaw Sakura | 62 szt,,,,,NaN,NaN,NaN,NaN
138,Zestaw Startowy | 16 szt,,,,,NaN,NaN,NaN,NaN
139,Zestaw Vibe | 32 szt,,,,,NaN,NaN,NaN,NaN
140,Zestaw Yami | 38 szt,,,,,NaN,NaN,NaN,NaN


In [402]:
# product_costs_mapping.to_csv(Path.cwd().parent / 'data' / 'mappings' / '07.2026' /'product_costs_mapping.csv', index=False)

## 5. Join Sales and Costs
Load the reviewed mapping and join it to sales using original product names.

In [403]:
costs_mapping = pd.read_csv(Path().cwd().parent / 'data' / 'mappings' / '07.2026' / 'product_costs_mapping.csv')

In [404]:
costs_mapping

,product_name_from_report,product_name_from_calculations_accepted,product_name_unified,product_category_accepted,product_cost_accepted,vat_rate_accepted,name_from_my_calculations,product_category_from_my_calculations,product_cost_my_calculations,vat_rate_my_calculations
0,Atlantic Wave roll | 8 szt,Atlantic Wave,Atlantic Wave roll | 8 szt,ROLKI AUTORSKIE,13.87,23.0,Akami Tamago roll,ROLKI AUTORSKIE,13.43,8.0
1,Awokado fresh roll | 8 szt,Awokado fresh roll,Awokado fresh roll | 8 szt,WEGE MENU,9.53,8.0,Atlantic Wave,ROLKI AUTORSKIE,13.87,23.0
2,Black dragon roll | 8 szt,Black dragon roll,Black dragon roll | 8 szt,ROLKI AUTORSKIE,18.29,8.0,Awokado delux,ROLKI AUTORSKIE,12.37,8.0
3,California KRAB-MIX | 8 szt,California KRAB,California KRAB-MIX | 8 szt,CALIFORNIA,7.71,8.0,Awokado fresh roll,WEGE MENU,9.53,8.0
4,California KREWETKA GOTOWANA | 8 szt,Calif krewetka 16/20,California KREWETKA GOTOWANA | 8 szt,CALIFORNIA,15.05,23.0,B. Wege,SUSHI BURGER,8.54,8.0
...,...,...,...,...,...,...,...,...,...,...
137,Zestaw Vibe | 32 szt,Vibe set | 32 szt,Vibe Set | 32 szt,ZESTAWY,50.77,8.0,NaN,NaN,NaN,NaN
138,Zestaw Yami | 38 szt,Yami set | 38 szt (Black Friday),Zestaw Yami | 38 szt,ZESTAWY,45.83,8.0,NaN,NaN,NaN,NaN
139,Śliwkowy,Śliwkowy,Śliwkowy,DODATKI,1.00,8.0,NaN,NaN,NaN,NaN
140,Zapieczona rolka ŁOSOŚ GRILLOWANY | 8 szt,Zapieczona rolka ŁOSOŚ GRILLOWANY | 8 szt,Zapieczona rolka ŁOSOŚ GRILLOWANY | 8 szt,ROLKI ZAPIECZONE,11.83,8.0,NaN,NaN,NaN,NaN


In [405]:
costs_table = costs_mapping[['product_name_from_report', 'product_name_from_calculations_accepted', 'product_name_unified', 'product_category_accepted', 'product_cost_accepted', 'vat_rate_accepted']]

In [406]:
costs_table

,product_name_from_report,product_name_from_calculations_accepted,product_name_unified,product_category_accepted,product_cost_accepted,vat_rate_accepted
0,Atlantic Wave roll | 8 szt,Atlantic Wave,Atlantic Wave roll | 8 szt,ROLKI AUTORSKIE,13.87,23.0
1,Awokado fresh roll | 8 szt,Awokado fresh roll,Awokado fresh roll | 8 szt,WEGE MENU,9.53,8.0
2,Black dragon roll | 8 szt,Black dragon roll,Black dragon roll | 8 szt,ROLKI AUTORSKIE,18.29,8.0
3,California KRAB-MIX | 8 szt,California KRAB,California KRAB-MIX | 8 szt,CALIFORNIA,7.71,8.0
4,California KREWETKA GOTOWANA | 8 szt,Calif krewetka 16/20,California KREWETKA GOTOWANA | 8 szt,CALIFORNIA,15.05,23.0
...,...,...,...,...,...,...
137,Zestaw Vibe | 32 szt,Vibe set | 32 szt,Vibe Set | 32 szt,ZESTAWY,50.77,8.0
138,Zestaw Yami | 38 szt,Yami set | 38 szt (Black Friday),Zestaw Yami | 38 szt,ZESTAWY,45.83,8.0
139,Śliwkowy,Śliwkowy,Śliwkowy,DODATKI,1.00,8.0
140,Zapieczona rolka ŁOSOŚ GRILLOWANY | 8 szt,Zapieczona rolka ŁOSOŚ GRILLOWANY | 8 szt,Zapieczona rolka ŁOSOŚ GRILLOWANY | 8 szt,ROLKI ZAPIECZONE,11.83,8.0


In [407]:
# Keep the original report name as the join key.
costs_table = costs_table.drop('product_name_from_calculations_accepted', axis=1).rename(columns={
    'product_name_from_report': 'product_name',
    'product_category_accepted': 'product_category',
    'product_cost_accepted': 'product_cost',
    'vat_rate_accepted': 'vat_rate'
})

In [409]:
costs_table

,product_name,product_name_unified,product_category,product_cost,vat_rate
0,Atlantic Wave roll | 8 szt,Atlantic Wave roll | 8 szt,ROLKI AUTORSKIE,13.87,23.0
1,Awokado fresh roll | 8 szt,Awokado fresh roll | 8 szt,WEGE MENU,9.53,8.0
2,Black dragon roll | 8 szt,Black dragon roll | 8 szt,ROLKI AUTORSKIE,18.29,8.0
3,California KRAB-MIX | 8 szt,California KRAB-MIX | 8 szt,CALIFORNIA,7.71,8.0
4,California KREWETKA GOTOWANA | 8 szt,California KREWETKA GOTOWANA | 8 szt,CALIFORNIA,15.05,23.0
...,...,...,...,...,...
137,Zestaw Vibe | 32 szt,Vibe Set | 32 szt,ZESTAWY,50.77,8.0
138,Zestaw Yami | 38 szt,Zestaw Yami | 38 szt,ZESTAWY,45.83,8.0
139,Śliwkowy,Śliwkowy,DODATKI,1.00,8.0
140,Zapieczona rolka ŁOSOŚ GRILLOWANY | 8 szt,Zapieczona rolka ŁOSOŚ GRILLOWANY | 8 szt,ROLKI ZAPIECZONE,11.83,8.0


In [410]:
sales_final = sales.merge(
    right=costs_table,
    how='left',
    on='product_name',
    validate='many_to_one'
)

In [412]:
sales_final

,product_name,total_sales,total_revenue,date,product_name_unified,product_category,product_cost,vat_rate
0,Futomaki ŁOSOŚ GRILLOWANY | 6 szt,12,456.0,2026-07-01,Futomaki ŁOSOŚ GRILLOWANY | 6 szt,FUTOMAKI,7.51,8.0
1,Wasabi,1,2.0,2026-07-01,Wasabi,DODATKI,0.30,8.0
2,Krabs - Burger,1,25.0,2026-07-01,Krabs - Burger,SUSHI BURGER,12.27,8.0
3,Sos teriyaki 30ml,1,3.0,2026-07-01,Sos teriyaki 30ml,DODATKI,1.00,8.0
4,Sos sojowy 30ml,2,6.0,2026-07-01,Sos sojowy 30ml,DODATKI,0.50,8.0
...,...,...,...,...,...,...,...,...
797,Futomaki KREWETKA PANKO | 6 szt,1,40.0,2026-07-31,Futomaki KREWETKA PANKO | 6 szt,FUTOMAKI,8.72,23.0
798,Zapieczona rolka ŁOSOŚ SUROWY | 8 szt,1,60.0,2026-07-31,Zapieczona rolka ŁOSOŚ SUROWY | 8 szt,ROLKI ZAPIECZONE,12.11,8.0
799,Fuji roll | 8 szt,1,65.0,2026-07-31,Fuji roll | 8 szt,ROLKI AUTORSKIE,14.86,8.0
800,Zapieczona rolka FELIX | 8 szt (NOWOŚĆ),1,60.0,2026-07-31,Zapieczona rolka FELIX | 8 szt,ROLKI ZAPIECZONE,10.83,8.0


## 6. Revenue Summary
Summarize quantities and revenue by original product name.

In [413]:
sales_by_product = sales_final.groupby(by='product_name')[['total_sales', 'total_revenue']].sum().sort_values(by='total_revenue', ascending=False)

In [414]:
sales_by_product

,total_sales,total_revenue
product_name,,
Futomaki ŁOSOŚ GRILLOWANY | 6 szt,216,8217.0
PROMO 1+1=3 Sushi burgery,58,6960.0
Futomaki FRESH ROLL | 6 szt,101,3535.0
Futomaki KALMAR PANKO | 6 szt,98,3136.0
Zestaw Vibe | 32 szt,15,3000.0
...,...,...
Fanta truskawka 0.33l,1,8.0
Fanta jagoda 0.33l,1,8.0
Pomocnik do pałeczek,5,5.0


In [415]:
sales_by_product['total_revenue'].sum()

np.float64(66539.5)

In [416]:
sales_final['product_name'].to_list()

['Futomaki ŁOSOŚ GRILLOWANY | 6 szt',
 'Wasabi',
 'Krabs - Burger',
 'Sos teriyaki 30ml',
 'Sos sojowy 30ml',
 'Zestaw Philadelphia | 24 szt',
 'Wasabi',
 'Zestaw Philadelphia + Krabs Burger -50%',
 'Sos sojowy',
 'California KREWETKA PANKO | 8 szt',
 'Zestaw Hikari | 48 szt',
 'Sos spicy majo 30ml',
 'Vibe roll ŁOSOŚ SUROWY | 8 szt',
 'Futomaki ŁOSOŚ SUROWY | 6 szt',
 'Futomaki KALMAR PANKO | 6 szt',
 'Futomaki GRZYBY SHIITAKE | 6 szt',
 'Futomaki KREWETKA PANKO | 6 szt',
 'Krabs - Burger',
 'Krewetki w tempurze | 6 szt',
 'Atlantic Wave roll | 8 szt',
 'Futomaki FRESH ROLL | 6 szt',
 'Sos teriyaki 30ml',
 'Futomaki WARZYWA W TEMPURZE | 6 szt',
 'Philadelphia mix | 8 szt',
 'Hosomaki KREWETKA PANKO | 8 szt',
 'Duo set | 28 szt',
 'Futomaki KREWETKA EBI | 6 szt',
 'Pomocnik do pałeczek',
 'Futomaki TATAR Z ŁOSOSIA | 6 szt',
 'Krewetki w tempurze | 6 szt',
 'Miso z łososiem | 350 ml',
 'Zestaw Vibe | 32 szt',
 'Zestaw Miyagi | 30 szt',
 'Sushi burger KREWETKA PANKO',
 'Tempura fusion ro

## 7. Calculate Metrics
Calculate VAT, net revenue, ingredient costs, and estimated profit before commissions and other operating expenses.

In [417]:
sales_final

,product_name,total_sales,total_revenue,date,product_name_unified,product_category,product_cost,vat_rate
0,Futomaki ŁOSOŚ GRILLOWANY | 6 szt,12,456.0,2026-07-01,Futomaki ŁOSOŚ GRILLOWANY | 6 szt,FUTOMAKI,7.51,8.0
1,Wasabi,1,2.0,2026-07-01,Wasabi,DODATKI,0.30,8.0
2,Krabs - Burger,1,25.0,2026-07-01,Krabs - Burger,SUSHI BURGER,12.27,8.0
3,Sos teriyaki 30ml,1,3.0,2026-07-01,Sos teriyaki 30ml,DODATKI,1.00,8.0
4,Sos sojowy 30ml,2,6.0,2026-07-01,Sos sojowy 30ml,DODATKI,0.50,8.0
...,...,...,...,...,...,...,...,...
797,Futomaki KREWETKA PANKO | 6 szt,1,40.0,2026-07-31,Futomaki KREWETKA PANKO | 6 szt,FUTOMAKI,8.72,23.0
798,Zapieczona rolka ŁOSOŚ SUROWY | 8 szt,1,60.0,2026-07-31,Zapieczona rolka ŁOSOŚ SUROWY | 8 szt,ROLKI ZAPIECZONE,12.11,8.0
799,Fuji roll | 8 szt,1,65.0,2026-07-31,Fuji roll | 8 szt,ROLKI AUTORSKIE,14.86,8.0
800,Zapieczona rolka FELIX | 8 szt (NOWOŚĆ),1,60.0,2026-07-31,Zapieczona rolka FELIX | 8 szt,ROLKI ZAPIECZONE,10.83,8.0


In [418]:
sales_final['unit_price_gross'] = sales_final['total_revenue'] / sales_final['total_sales']

In [419]:
sales_final['unit_vat'] = sales_final['unit_price_gross'] * sales_final['vat_rate'] / (100 + sales_final['vat_rate'])

In [420]:
sales_final['unit_price_net'] = sales_final['unit_price_gross'] - sales_final['unit_vat']

In [421]:
sales_final['unit_profit_est'] = sales_final['unit_price_net'] - sales_final['product_cost']
sales_final['margin_rate'] = sales_final['unit_profit_est'] / sales_final['unit_price_net']
sales_final['food_cost_rate'] = sales_final['product_cost'] / sales_final['unit_price_net']
sales_final['total_revenue_net'] = sales_final['total_sales'] * sales_final['unit_price_net']
sales_final['total_revenue_gross'] = sales_final['total_sales'] * sales_final['unit_price_gross']
sales_final['total_cost_est'] = sales_final['total_sales'] * sales_final['product_cost']
sales_final['total_profit_est'] = sales_final['total_revenue_net'] - sales_final['total_cost_est']

In [422]:
sales_final

,product_name,total_sales,total_revenue,date,product_name_unified,product_category,product_cost,vat_rate,unit_price_gross,unit_vat,unit_price_net,unit_profit_est,margin_rate,food_cost_rate,total_revenue_net,total_revenue_gross,total_cost_est,total_profit_est
0,Futomaki ŁOSOŚ GRILLOWANY | 6 szt,12,456.0,2026-07-01,Futomaki ŁOSOŚ GRILLOWANY | 6 szt,FUTOMAKI,7.51,8.0,38.0,2.814815,35.185185,27.675185,0.786558,0.213442,422.222222,456.0,90.12,332.102222
1,Wasabi,1,2.0,2026-07-01,Wasabi,DODATKI,0.30,8.0,2.0,0.148148,1.851852,1.551852,0.838000,0.162000,1.851852,2.0,0.30,1.551852
2,Krabs - Burger,1,25.0,2026-07-01,Krabs - Burger,SUSHI BURGER,12.27,8.0,25.0,1.851852,23.148148,10.878148,0.469936,0.530064,23.148148,25.0,12.27,10.878148
3,Sos teriyaki 30ml,1,3.0,2026-07-01,Sos teriyaki 30ml,DODATKI,1.00,8.0,3.0,0.222222,2.777778,1.777778,0.640000,0.360000,2.777778,3.0,1.00,1.777778
4,Sos sojowy 30ml,2,6.0,2026-07-01,Sos sojowy 30ml,DODATKI,0.50,8.0,3.0,0.222222,2.777778,2.277778,0.820000,0.180000,5.555556,6.0,1.00,4.555556
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
797,Futomaki KREWETKA PANKO | 6 szt,1,40.0,2026-07-31,Futomaki KREWETKA PANKO | 6 szt,FUTOMAKI,8.72,23.0,40.0,7.479675,32.520325,23.800325,0.731860,0.268140,32.520325,40.0,8.72,23.800325
798,Zapieczona rolka ŁOSOŚ SUROWY | 8 szt,1,60.0,2026-07-31,Zapieczona rolka ŁOSOŚ SUROWY | 8 szt,ROLKI ZAPIECZONE,12.11,8.0,60.0,4.444444,55.555556,43.445556,0.782020,0.217980,55.555556,60.0,12.11,43.445556
799,Fuji roll | 8 szt,1,65.0,2026-07-31,Fuji roll | 8 szt,ROLKI AUTORSKIE,14.86,8.0,65.0,4.814815,60.185185,45.325185,0.753095,0.246905,60.185185,65.0,14.86,45.325185
800,Zapieczona rolka FELIX | 8 szt (NOWOŚĆ),1,60.0,2026-07-31,Zapieczona rolka FELIX | 8 szt,ROLKI ZAPIECZONE,10.83,8.0,60.0,4.444444,55.555556,44.725556,0.805060,0.194940,55.555556,60.0,10.83,44.725556


## 8. Export for Power BI
Select output columns, use unified product names, and save the CSV.

In [423]:
sales_final = sales_final[[
    'date', 'product_name', 'product_name_unified', 'product_category', 'total_sales', 'total_revenue', 'product_cost', 'vat_rate', 'unit_price_gross', 
    'unit_vat', 'unit_price_net', 'unit_profit_est', 'margin_rate', 'food_cost_rate', 'total_revenue_net', 'total_revenue_gross', 'total_cost_est',
    'total_profit_est',
]]

In [424]:
sales_final

,date,product_name,product_name_unified,product_category,total_sales,total_revenue,product_cost,vat_rate,unit_price_gross,unit_vat,unit_price_net,unit_profit_est,margin_rate,food_cost_rate,total_revenue_net,total_revenue_gross,total_cost_est,total_profit_est
0,2026-07-01,Futomaki ŁOSOŚ GRILLOWANY | 6 szt,Futomaki ŁOSOŚ GRILLOWANY | 6 szt,FUTOMAKI,12,456.0,7.51,8.0,38.0,2.814815,35.185185,27.675185,0.786558,0.213442,422.222222,456.0,90.12,332.102222
1,2026-07-01,Wasabi,Wasabi,DODATKI,1,2.0,0.30,8.0,2.0,0.148148,1.851852,1.551852,0.838000,0.162000,1.851852,2.0,0.30,1.551852
2,2026-07-01,Krabs - Burger,Krabs - Burger,SUSHI BURGER,1,25.0,12.27,8.0,25.0,1.851852,23.148148,10.878148,0.469936,0.530064,23.148148,25.0,12.27,10.878148
3,2026-07-01,Sos teriyaki 30ml,Sos teriyaki 30ml,DODATKI,1,3.0,1.00,8.0,3.0,0.222222,2.777778,1.777778,0.640000,0.360000,2.777778,3.0,1.00,1.777778
4,2026-07-01,Sos sojowy 30ml,Sos sojowy 30ml,DODATKI,2,6.0,0.50,8.0,3.0,0.222222,2.777778,2.277778,0.820000,0.180000,5.555556,6.0,1.00,4.555556
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
797,2026-07-31,Futomaki KREWETKA PANKO | 6 szt,Futomaki KREWETKA PANKO | 6 szt,FUTOMAKI,1,40.0,8.72,23.0,40.0,7.479675,32.520325,23.800325,0.731860,0.268140,32.520325,40.0,8.72,23.800325
798,2026-07-31,Zapieczona rolka ŁOSOŚ SUROWY | 8 szt,Zapieczona rolka ŁOSOŚ SUROWY | 8 szt,ROLKI ZAPIECZONE,1,60.0,12.11,8.0,60.0,4.444444,55.555556,43.445556,0.782020,0.217980,55.555556,60.0,12.11,43.445556
799,2026-07-31,Fuji roll | 8 szt,Fuji roll | 8 szt,ROLKI AUTORSKIE,1,65.0,14.86,8.0,65.0,4.814815,60.185185,45.325185,0.753095,0.246905,60.185185,65.0,14.86,45.325185
800,2026-07-31,Zapieczona rolka FELIX | 8 szt (NOWOŚĆ),Zapieczona rolka FELIX | 8 szt,ROLKI ZAPIECZONE,1,60.0,10.83,8.0,60.0,4.444444,55.555556,44.725556,0.805060,0.194940,55.555556,60.0,10.83,44.725556


In [425]:
sales_final = sales_final.drop('product_name', axis=1).rename({'product_name_unified': 'product_name'}, axis=1)

In [427]:
sales_final.to_csv(Path().cwd().parent / 'data' / 'output' / 'sales_07_2026.csv', index=False)